In [0]:
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.types import TimestampType, IntegerType, StringType

In [0]:
"""
create lookup table dataframe and verify
Note that csv imports as string unless a schema is created and applied
"""

df = spark.read.csv('/Volumes/workspace/00_landing/data_sources/lookup/taxi_zone_lookup.csv', header=True)

In [0]:
"""
start cleaning and enhancing data
change column names to lowercase where needed
create a current_timestamp column called 'effective_date
create a new column alias to 'end_date', fill with null values but of type Timestamp
'lit(None).cast(TimestampType())' will create null values but in a Timestamp column
"""
df = df.select(
    col("LocationID").cast(IntegerType()).alias("location_id"),
    col("Borough").alias("borough"),
    col("Zone").alias("zone"),
    col("service_zone"), 
    current_timestamp().alias("effective_date"),
    lit(None).cast(TimestampType()).alias("end_date")
)

In [0]:
# fixed point-in-time used to "close" any changed active recods
# using python timestamp ensures the exact same value is written and can be referenced if needed
end_timestamp = datetime.now()

# Load the SCD2 delta table
dt = DeltaTable.forName(spark, "02_silver.taxi_zone_lookup")

In [0]:
"""
PASS 1 Merge:  Close any active rows whose tracked attributes changed

Match only the active target row (end_date = NULL), with the same business key
If ant tracked column differs, set the end_date to end_timestamp to retire/close that record
"""

dt.alias("target").\
    merge(
        source = df.alias("source"),
        condtion = "target.location_id = source.location_id AND target.end_date IS NULL AND (target.borough != source.borough OR target.zone != source.zone OR target.service_zone != source.service_zone)"
    ).\
    whenMatchedUpdate(
        set = {"target.end_date": lit(end_timestamp).cast(TimestampType())}
    ).execute()

In [0]:
"""
PASS 2 Merge: Insert new versions

Now insert a row for:
    1. keys we just closed in PASS 1 (no longer active)
    2. brand new keys not present in the target

NOTE:  'end_timestamp' set in PASS 1

"""
insert_id_list = [row.location_id for row in dt.toDf().filter(f"end_date = '{end_timestamp}'").select("location_id").collect()]

if len(insert_id_list) == 0:
    print("No updated records to insert")
else:
    dt.alias("target").\
        merge(
            source = df.alias("source"),
            condition = f"source.location_id not in ({', '.join(map(str, insert_id_list))})"
        ).\
        whenNotMatchedInsert(
            values = {
                "target.location_id": "source.location_id",
                "target.borough": "source.borough",
                "target.zone": "source.zone",
                "target.service_zone": "source.service_zone",
                "target.effective_date": current_timestamp(),
                "target.end_date": lit(None).cast(TimestampType())
            }
        ).execute()

In [0]:
"""
Pass 3
insert brand new rows which have no historical row in target table
"""

dt.alias("target").\
    merge(
        source = df.alias("source"),
        condition = "target.location_id = source.location_id"
    ).\
    whenNotMatchedInsert(
        values = {
            "target.location_id": "source.location_id",
            "target.borough": "source.borough",
            "target.zone": "source.zone",
            "target.service_zone": "source.service_zone",
            "target.effective_date": current_timestamp(),
            "target.end_date": lit(None).cast(TimestampType())
        }
    ).execute()